# 23e · Does a larger Double DQN help? (network-size check on the E23d setup)

E23d (long / flat Double DQN, enriched factors, a decision every 60 bars, checkpoint chosen by V1 skill, judged once
on V2) used E16's tuned network: **small**, 10 -> 128 -> 128 -> 3 (about 18.6k parameters). E16's tuning had already
preferred it over the two larger options on validation, and E21 showed the small network's Q-values already follow a
planted signal. This notebook checks directly whether more capacity changes the outcome.

**Only `net_dims` changes**; everything else is E23d (E16 hyperparameters, state A, enriched cross-fitted factors,
`allow_short=False`, `step_gap=60`, 1M steps, checkpoint by V1 skill with >= 5 V1 trades, stay flat unless V1 skill > 0):

| network | hidden layers |
|---|---|
| small (E23d, reused) | 128, 128 |
| medium | 256, 256, 128 |
| notebook | 256, 256, 128, 128, 64 |

5 seeds per new network = **10 runs**, about 3 h on a T4; runs resume after a disconnect. **Test and stress_bear are
not loaded.** Writes `thesis_rl/initiation/e23e_net_size/` and `report/e23e_*`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "23e_net_size")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from rltrade import halves as HV
from rltrade.agents.nets import mlp
from rltrade.env import EnvConfig

INIT = L.root / "initiation"
OUT, REP, E23D = INIT / "e23e_net_size", INIT / "report", INIT / "e23d_long_flat_g60"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
FACTORS = INIT / "e23_factors" / "crossfit_mean3"
assert (FACTORS / "factors_train.npy").exists() and (FACTORS / "factors_val.npy").exists(), f"missing {FACTORS} (run 23b)"
DATA, CUTS = HV.write_halves(PREP, FACTORS, LOCAL / "e23e" / "enriched")

hp = json.loads((INIT / "agents" / "tuning" / "best_hp.json").read_text())       # E16's tuned Double DQN
BASE_HP, ENV_OVER = hp["agent_hp"], {**hp["env"], "step_gap": 60, "allow_short": False}
CFG = EnvConfig(state="A", **ENV_OVER)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
NETS = {"small": tuple(BASE_HP["net_dims"]), "medium": (256, 256, 128), "notebook": (256, 256, 128, 128, 64)}
NEW_NETS = ("medium", "notebook")                                                 # small = E23d, reused
SEEDS, TOTAL_STEPS, EVAL_EVERY, MIN_TRADES = (0, 1, 2, 3, 4), 1_000_000, 100_000, 5
def run_name(net, seed): return f"enriched_longflat_g60_{net}_s{seed}"
def n_params(dims): return sum(p.numel() for p in mlp(CFG.state_dim, dims, 3).parameters())
PARAMS = {k: n_params(v) for k, v in NETS.items()}
print("networks:", {k: (v, f"{PARAMS[k]:,} params") for k, v in NETS.items()})
print("base agent:", BASE_HP, ENV_OVER, "| device", DEV, "| V1 / V2 rows:", CUTS["v1"]["rows"], CUTS["v2"]["rows"])

## 1 · Reference: buy and hold

In [ ]:
REF = HV.reference(DATA, CFG, DEV)
BH = {s: m["sharpe"] for s, m in REF.items()}
BH_RET = {s: m["cum_return_pct"] for s, m in REF.items()}
print("buy and hold (exact, 60-bar decisions): " + ", ".join(f"{s} Sharpe {BH[s]:+.2f} ({BH_RET[s]:+.1f} %)" for s in HV.SPLITS))

## 2 · Train the two larger networks (resumes)

In [ ]:
t0 = time.time()
for net in NEW_NETS:
    agent_hp = {**BASE_HP, "net_dims": NETS[net]}
    for seed in SEEDS:
        HV.train_select(run_name(net, seed), DATA, OUT, seed, agent_hp, ENV_OVER, TOTAL_STEPS, EVAL_EVERY, device=DEV,
                        min_trades=MIN_TRADES, select_by="skill")
print(f"training done ({(time.time() - t0) / 60:.0f} min this session)")

## 3 · Results on V2, by network size (small = E23d)

In [ ]:
def load(net, seed):
    f = (E23D / "runs" / f"enriched_longflat_g60_s{seed}" if net == "small" else OUT / "runs" / run_name(net, seed)) / "result.json"
    return json.loads(f.read_text()) if f.exists() else None

def row(net, seed, r):
    out = {"network": net, "seed": seed}
    for by in ("skill", "sharpe"):
        sel = r["selections"][by]
        flat = (not sel) or sel["fallback"]
        out[f"{by} V2 Sharpe"] = 0.0 if flat else sel["v2"]["sharpe"]
        out[f"{by} V2 return %"] = 0.0 if flat else sel["v2"]["cum_return_pct"]
        out[f"{by} stayed flat"] = flat
        if by == "skill":
            out["skill V2 trades"] = 0 if flat else sel["v2"]["num_trades"]
            out["skill V2 max DD %"] = 0.0 if flat else sel["v2"]["max_drawdown_pct"]
            out["skill step"] = sel["steps"] if sel else None
            out["skill V1 skill %"] = sel["v1"]["skill_pct"] if sel else None
            out["skill ckpt V2 skill %"] = sel["v2"]["skill_pct"] if sel else None
            out["skill ckpt V2 in market %"] = sel["v2"]["exposure_pct"] if sel else None
    out["final V2 Sharpe"] = r["final"]["v2"]["sharpe"]
    out["best V1 skill over curve %"] = max((c["v1_skill_pct"] for c in r["curve"] if c["eligible"]), default=np.nan)
    out["train Sharpe at end"] = r["curve"][-1]["train_sharpe"]
    return out

RES = {(n, s): load(n, s) for n in NETS for s in SEEDS}
missing = [k for k, v in RES.items() if v is None]
if missing: print("WARNING missing runs (small = E23d runs must be on Drive):", missing)
runs = pd.DataFrame([row(n, s, r) for (n, s), r in RES.items() if r is not None])
runs.to_csv(OUT / "runs_summary.csv", index=False)
keys = ["skill V2 Sharpe", "skill V2 return %", "skill V2 trades", "skill ckpt V2 skill %", "sharpe V2 Sharpe",
        "final V2 Sharpe", "train Sharpe at end"]
agg = runs.groupby("network", sort=False)[keys].agg(["mean", "std"]).round(2)
agg.columns = [f"{a} {b}" for a, b in agg.columns]
agg.insert(0, "params", [PARAMS[n] for n in agg.index])
agg.insert(1, "runs", runs.groupby("network", sort=False).size().values)
pd.set_option("display.width", 250)
print("every run:\n" + runs.round(2).to_string(index=False))
print("\nmean and SD over seeds, by network:\n" + agg.to_string())
print(f"\nV2 buy and hold: Sharpe {BH['v2']:+.2f} ({BH_RET['v2']:+.1f} %)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
panels = (("train_sharpe", "train (chunked): Sharpe"), ("v1_skill_pct", "V1 (choice): skill %"), ("v2_sharpe", "V2 (judged): Sharpe"))
colours = {"small": "#6E7781", "medium": "#D9822B", "notebook": "#2F6DB5"}
for ax, (key, title) in zip(axes, panels):
    for (n, s), r in RES.items():
        if r is None: continue
        c = pd.DataFrame(r["curve"])
        ax.plot(c.steps / 1e6, c[key], lw=1.1, alpha=0.65, color=colours[n], label=n if s == SEEDS[0] else None)
    if key == "v2_sharpe": ax.axhline(BH["v2"], color="black", ls="--", lw=1, label="buy and hold")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title(title, fontsize=10); ax.set_xlabel("training steps (millions)"); ax.grid(alpha=0.3)
axes[2].legend(fontsize=8)
fig.suptitle("long / flat Double DQN, enriched factors: small vs larger networks (5 seeds each)", fontsize=11)
fig.tight_layout(); fig.savefig(REP / "e23e_net_size_curves.png", dpi=130); plt.show()

In [ ]:
(REP / "e23e_net_size.md").write_text(
    "# E23e: does a larger Double DQN help? (train + validation only)\n\n"
    "Same setup as E23d (long / flat, enriched cross-fitted factors, decision every 60 bars, checkpoint = best V1 skill "
    f"with >= {MIN_TRADES} V1 trades, judged once on V2, stay flat unless V1 skill > 0, {TOTAL_STEPS:,} steps); only the "
    "network changes. small = the E23d runs. Test and stress_bear not loaded.\n\n"
    "Networks: " + "; ".join(f"{k} {v} ({PARAMS[k]:,} params)" for k, v in NETS.items()) + "\n\n"
    f"V2 buy and hold: Sharpe {BH['v2']:+.2f} ({BH_RET['v2']:+.1f} %)\n\n"
    "## Mean and SD over seeds\n\n" + agg.reset_index().to_markdown(index=False)
    + "\n\n## Every run\n\n" + runs.round(3).to_markdown(index=False)
    + "\n\nFigure: e23e_net_size_curves.png\n", encoding="utf-8")
print("wrote", REP / "e23e_net_size.md")

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()